In [3]:
import plotly.graph_objects as go
import networkx as nx
import sys
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

def load_graph(tissue_name, threshold=0.5):
    graph_path = f"tissue_networks/{tissue_name.replace(' ', '_')}_network.gexf"
    G = nx.read_gexf(graph_path)
    G_filtered = G.copy()
    edges_to_remove = [(u, v) for u, v, data in G_filtered.edges(data=True) 
                    if data.get('weight', 0) < threshold]
    G_filtered.remove_edges_from(edges_to_remove)
    return G_filtered

def load_embeddings(tissue_name):
    embeddings = pd.read_csv(f"tissue_embeddings/2d-umap/{tissue_name}_embeddings_2d.csv")
    names = embeddings['node'].tolist()
    emb_matrix = embeddings.drop(columns=['node']).to_numpy()
    print(emb_matrix.shape)
    return names, emb_matrix

In [ ]:
#create and export spring_layout

def export_fr_layout(tissue_name, threshold):
    G = load_graph(tissue_name=tissue_name, threshold=threshold)
    pos = nx.spring_layout(G, seed=42)
    
    data = [{'node': node, 'x': x, 'y': y} for node, (x, y) in pos.items()]
    df = pd.DataFrame(data)
    filename = f'{tissue_name}_layout.csv'
    df.to_csv(filename, index=False)
    
    return filename
    
pos = export_fr_layout("Adipose_Subcutaneous", threshold = 0.95)
print(pos)

{'C1QC': array([-0.01428489,  0.1165935 ]), 'C1QB': array([-0.01303265,  0.11892113]), 'LCN10': array([-0.19047453, -0.00376556]), 'LCN6': array([-0.19022095, -0.00099023]), 'RPL31': array([-0.13768254, -0.02451047]), 'RPL32': array([-0.13592587, -0.02830593]), 'MYH11': array([ 0.02102263, -0.03360888]), 'CNN1': array([ 0.01935074, -0.03516727]), 'PRSS1': array([-0.07876622, -0.05801523]), 'PRSS2': array([-0.07882626, -0.05546193]), 'RPL19': array([-0.13466123, -0.03146009]), 'FCGR2C': array([-0.0520383,  0.0426865]), 'HSPA7': array([-0.05171503,  0.04493972]), 'HBB': array([-0.1060635 , -0.13853982]), 'HBA2': array([-0.10349395, -0.14031528]), 'COL1A2': array([-0.07490969,  0.02696369]), 'COL1A1': array([-0.07263587,  0.02620393]), 'ACTG2': array([ 0.02009925, -0.03351713]), 'RPS27A': array([ 0.0107493 , -0.09726939]), 'RPL21': array([ 0.01066747, -0.09963609]), 'RPL13A': array([-0.13280883, -0.03270761]), 'C1QA': array([-0.01343162,  0.11695105]), 'RPL7A': array([-0.79186714,  0.4926

In [ ]:
edge_x = []
edge_y = []
G = load_graph(tissue_name="Adipose_Subcutaneous", threshold=0.95)
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.append(x0)
    edge_x.append(x1)
    edge_x.append(None)
    edge_y.append(y0)
    edge_y.append(y1)
    edge_y.append(None)

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

node_x = []
node_y = []
for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    hoverinfo='text',
    text=list(G.nodes()),
    marker=dict(
        showscale=True,
        colorscale='YlGnBu',
        size=10,
        line_width=2))

fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(
                    title=f"Adipose_Subcutaneous Network",
                    showlegend=False,
                    hovermode='closest',
                    margin=dict(b=20,l=5,r=5,t=40),
                    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
                ))
fig.show()

In [ ]:

def visualize_community_graph(tissue_name, threshold=0.5, resolution=1.0):
    G = load_graph(tissue_name=tissue_name, threshold=threshold)
    names, embeddings_2d = load_embeddings(tissue_name)
    print(f"loaded embeddings for {tissue_name}.")
    print(f"loading graph for {tissue_name}...")

    name_to_embedding = {name: emb for name, emb in zip(names, embeddings_2d)}

    communities = nx.community.louvain_communities(G, weight='weight', seed=42, resolution=resolution)
    print(f"Detected {len(communities)} communities in the graph.")
    community_graph = nx.Graph()
    pos = {}  

    for i, community in enumerate(communities):
        community_size = len(community)
        community_graph.add_node(i, size=community_size)
        
        community_embeddings = []
        for node in community:
            if node in name_to_embedding:
                community_embeddings.append(name_to_embedding[node])
        
        if community_embeddings:
            avg_embedding = np.mean(community_embeddings, axis=0)
            pos[i] = (avg_embedding[0], avg_embedding[1])
        else:
            pos[i] = (0, 0)
        
        for j in range(i + 1, len(communities)):
            weight = sum(1 for u in community for v in communities[j] if G.has_edge(u, v))
            if weight > 0:
                community_graph.add_edge(i, j, weight=weight)

    edge_x = []
    edge_y = []
    for edge in community_graph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.append(x0)
        edge_x.append(x1)
        edge_x.append(None)
        edge_y.append(y0)
        edge_y.append(y1)
        edge_y.append(None)

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        mode='lines')
    # Nodes
    node_x = []
    node_y = []
    node_sizes = []
    for node in community_graph.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_sizes.append(community_graph.nodes[node]['size'])

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        text=[f"Community {i}: {size} nodes" for i, size in enumerate(node_sizes)],
        marker=dict(
            showscale=True,
            colorscale='YlGnBu',
            size=node_sizes,  # Size based on community size
            sizemode='area',
            sizeref=2.*max(node_sizes)/(40.**2),  # Scale for visibility
            sizemin=4,
            line_width=2))

    fig = go.Figure(data=[edge_trace, node_trace],
        layout=go.Layout(
            title=f"{tissue_name} Network Community Graph",
            showlegend=False,
            hovermode='closest',
            margin=dict(b=20,l=5,r=5,t=40),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
        ))
    
    print(f"{len(max(nx.connected_components(G), key=len))} nodes in largest connected component.")
    print(G.number_of_nodes())
    print(f"number of connected components: {nx.number_connected_components(G)}")

    fig.show()
tissue_name = "Liver"
threshold = 0.0
visualize_community_graph(tissue_name, threshold)